# 260416 ETF 데이터 전처리 + RAG용 Document 생성

> **Week 6 Day 3** | 수업 노트북 2개를 하나로 통합 (Part 1: 데이터 분석 / Part 2: RAG 구축)

---

## 오늘 배우는 것

| 파트 | 내용 | 비유 |
|------|------|------|
| Part 1 | ETF 가격 수집, 결측치/이상치 처리, 이동평균, 트렌드 분해, 상관관계, 성과 지표 | 식재료 손질 + 맛 평가 |
| Part 2 | LLM으로 Document 생성, FAISS 벡터 스토어, BM25 키워드 검색, 메타데이터 필터링 | 손질된 재료로 요리(RAG 챗봇) 만들기 |

### 핵심 흐름
```
실시간 가격 데이터 수집 (FinanceDataReader)
    -> 결측치 처리 (ffill / interpolation / rolling)
    -> 이상치 탐지 (IQR / Z-score)
    -> Feature 계산 (return, volatility, MDD, Sharpe)
    -> LLM으로 Description 생성
    -> Document + metadata 구성
    -> FAISS 벡터 스토어 + BM25 키워드 검색
    -> 메타데이터 필터링 (카테고리, 수수료 등)
```

**왜 이 순서?** 단순한 ETF 이름/카테고리만으로는 "돈 벌고 싶은데 뭐 사야 돼?" 같은 질문에 답할 수 없다.  
수치 데이터(수익률, 변동성 등)를 LLM이 읽을 수 있는 자연어 Document로 변환해야 RAG가 제대로 동작한다.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w6_finance_rag/llm_260416_etf_data_rag.ipynb)

---
## Colab 환경 설정

In [ ]:
# === Colab 사용 시 아래 주석 해제 ===
# !pip install openai langchain-openai langchain-community faiss-cpu python-dotenv gradio
# !pip install finance-datareader statsmodels kiwipiepy rank_bm25

# import os
# from google.colab import userdata
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

---
# Part 1: ETF 데이터 수집 & 전처리

> 비유: RAG 챗봇을 만들기 전에 "식재료 손질"을 먼저 해야 한다.  
> 날것의 주가 데이터에는 빠진 날(결측치), 튀는 값(이상치)이 있어서 그대로 쓰면 분석 결과가 엉망이 된다.

In [ ]:
import os, json, math
from datetime import datetime, timedelta
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
# load_dotenv()

plt.rcParams['font.family'] = 'NanumGothic'  # Colab에서는 'NanumBarunGothic'
plt.rcParams['axes.unicode_minus'] = False

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini")

### 1-1. 한국 ETF 시장 현황 파악

In [ ]:
# 한국 ETF 카테고리별 종목 수 (대략적)
categories = {
    "국내주식": 145, "해외주식": 120, "채권": 85,
    "섹터": 95, "원자재": 25, "부동산": 15, "기타": 30,
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.pie(categories.values(), labels=categories.keys())
ax2.barh(list(categories.keys()), list(categories.values()))
plt.show()

### 1-2. 샘플 ETF 데이터로 리스크-수익률 산점도 그리기

> 비유: 각 ETF를 "위험(x축) vs 보상(y축)" 지도에 찍어보는 것.  
> 오른쪽 위 = 높은 위험 + 높은 수익, 왼쪽 아래 = 안전하지만 수익 낮음.

In [ ]:
etf_universe = [
    {"name": "KODEX 200", "cat": "국내주식", "ret_1y": 12.3, "risk": 15.2},
    {"name": "TIGER S&P500", "cat": "해외주식", "ret_1y": 18.5, "risk": 13.8},
    {"name": "KODEX 배당가치", "cat": "배당", "ret_1y": 8.2, "risk": 10.5},
    {"name": "TIGER 반도체", "cat": "섹터", "ret_1y": 35.2, "risk": 28.4},
    {"name": "KODEX 국고채3년", "cat": "채권", "ret_1y": 3.5, "risk": 2.1},
    {"name": "KODEX 골드선물", "cat": "원자재", "ret_1y": 15.8, "risk": 16.3},
    {"name": "KODEX 2차전지", "cat": "섹터", "ret_1y": -5.2, "risk": 32.1},
    {"name": "TIGER 단기통안채", "cat": "채권", "ret_1y": 3.2, "risk": 0.8},
]

df_etf = pd.DataFrame(etf_universe)
df_etf

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for cat in df_etf['cat'].unique():
    sub = df_etf[df_etf['cat'] == cat]
    ax.scatter(sub['risk'], sub['ret_1y'], label=cat)
    for _, row in sub.iterrows():
        ax.annotate(row['name'][:8], (row['risk'], row['ret_1y']))

ax.set_xlabel('리스크 (%)')
ax.set_ylabel('1년 수익률 (%)')
ax.set_title('ETF 리스크-수익률 산점도')
ax.axhline(y=10, color='gray', linestyle='--', alpha=0.5)
ax.legend()
plt.show()

### 1-3. dataclass로 ETF 스키마 정의하기

> 비유: dataclass = ETF 정보의 "주민등록증 양식".  
> 필수 항목(ticker, name)이 빠지면 만들 수 없고, 수수료가 범위를 벗어나면 거부된다.  
> `__post_init__`은 양식 제출 시 자동으로 돌아가는 "유효성 검사기".

In [ ]:
@dataclass
class ETFSchema:
    ticker: str            # 6자리 종목코드
    name: str
    category: str
    expense_ratio: float   # 수수료 (%)
    risk_level: str = "중간"
    aum_billion: float = 0.0
    description: str = ""
    keywords: List[str] = field(default_factory=list)

    def __post_init__(self):
        # 수수료 범위 검증 (0~5%)
        if self.expense_ratio < 0 or self.expense_ratio > 5:
            raise ValueError(f"수수료 범위 오류: {self.expense_ratio}")
        # 리스크 등급 검증
        valid_risks = ["매우낮음", "낮음", "중간", "높음", "매우높음"]
        if self.risk_level not in valid_risks:
            raise ValueError(f"리스크 등급 오류: {self.risk_level}")

In [ ]:
# 정상 생성
etf = ETFSchema(
    ticker="069500", name="KODEX 200", category="국내주식",
    expense_ratio=1.5, keywords=['코스피', '대형주', '인덱스'])
etf

In [ ]:
# dataclass -> dict -> JSON 변환 (RAG Document 만들 때 유용)
etf_dict = asdict(etf)
etf_json = json.dumps(etf_dict, ensure_ascii=False, indent=2)
print(etf_json)

In [ ]:
# JSON -> dataclass 역변환
ETFSchema(**json.loads(etf_json))

In [ ]:
# 유효성 검증 테스트: 수수료 오류, 리스크 등급 오류 잡아내기
test_cases = [
    {"ticker": "A", "name": "정상", "category": "채권",
     "expense_ratio": 0.05, "risk_level": "낮음"},
    {"ticker": "B", "name": "수수료오류", "category": "주식",
     "expense_ratio": -0.5, "risk_level": "중간"},
    {"ticker": "C", "name": "리스크오류", "category": "원자재",
     "expense_ratio": 0.3, "risk_level": "최고"},
]

for tc in test_cases:
    try:
        e = ETFSchema(**tc)
        print(f" {tc['name']}: 생성 성공")
    except ValueError as err:
        print(f" {tc['name']} : {err}")

In [ ]:
# 수익률 데이터 스키마 (getattr + for 루프로 여러 필드 한번에 검증)
@dataclass
class ETFReturns:
    ticker: str
    return_1m: float
    return_3m: float
    return_1y: float
    return_3y: float

    def __post_init__(self):
        for field_name in ['return_1m', 'return_3m', 'return_1y', 'return_3y']:
            value = getattr(self, field_name)
            if value < -100 or value > 500:
                raise ValueError(f"{field_name} 오류: {value}")

# 테스트: 이 줄의 주석을 해제하면 ValueError 발생
# ETFReturns("123456", -200.1, 5.4, 12.3, 28.5)

### 1-4. FinanceDataReader로 실시간 가격 수집

> 비유: FinanceDataReader는 주식시장의 "택배 기사".  
> ticker(종목코드)와 기간을 알려주면 매일의 시가/고가/저가/종가/거래량을 배달해 준다.

In [ ]:
# !pip install finance-datareader  # 설치 안 되어 있으면 주석 해제
import FinanceDataReader as fdr

# 한국 상장 ETF 목록 가져오기
etf_list = fdr.StockListing('ETF/KR')
print(f"한국 상장 ETF 수: {len(etf_list)}")
etf_list.head(3)

In [ ]:
# 특정 ETF 1년치 가격 가져오기
def collect_etf_price(ticker, days=365):
    """ticker에 해당하는 ETF의 가격 데이터를 수집"""
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - timedelta(days=days)).strftime('%Y-%m-%d')
    try:
        df = fdr.DataReader(ticker, start_date, end_date)
        if len(df) > 0:
            return {'status': 'real', 'data': df, 'ticker': ticker}
    except Exception:
        pass

result = collect_etf_price('069500')
price_df = result['data']
print(f"수집된 거래일 수: {len(price_df)}")
price_df.tail(3)

### 1-5. 결측치 처리: 3가지 방법 비교

> 비유: 출석부에 빠진 날이 있을 때 어떻게 메울 것인가?
> - **Forward Fill**: 어제 값 그대로 쓰기 ("어제랑 같겠지")
> - **Interpolation**: 앞뒤 값의 중간값 ("양쪽 평균이겠지")
> - **Rolling Mean**: 최근 N일 평균 ("최근 흐름을 반영하자")

주말/휴장일은 row 자체가 없으므로 문제 없고, 간혹 특정 컬럼만 NaN인 경우를 처리해야 한다.

In [ ]:
# 인위적으로 5% 결측치 생성 (실습용)
sample_df = price_df.copy()
mask = np.random.random(len(sample_df)) < 0.05
sample_df.loc[mask, 'Close'] = np.nan
print(f"결측치 수: {sample_df['Close'].isna().sum()} / {len(sample_df)}")

In [ ]:
# 방법 1: Forward Fill (이전 값으로 채우기)
ffill_df = sample_df.copy()
ffill_df['Close'] = ffill_df['Close'].ffill()

# 방법 2: Linear Interpolation (선형 보간)
interp_df = sample_df.copy()
interp_df['Close'] = interp_df['Close'].interpolate(method='linear')

# 방법 3: Rolling Mean (이동 평균)
rolling_df = sample_df.copy()
rolling_mean = rolling_df['Close'].rolling(5, min_periods=1).mean()
rolling_df['Close'] = rolling_df['Close'].fillna(rolling_mean)

# 3가지 비교 시각화
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, df_filled, title in zip(
    axes, [ffill_df, interp_df, rolling_df],
    ['Forward Fill', 'Interpolation', 'Rolling Mean']
):
    ax.plot(sample_df.index, sample_df['Close'], 'ro', markersize=3, alpha=0.3, label='missing')
    ax.plot(df_filled.index, df_filled['Close'], label=title)
    ax.set_title(title)
plt.tight_layout()
plt.show()

### 1-6. 이상치 탐지: IQR vs Z-score

> 비유: 시험 점수에서 "이건 좀 이상한데?" 싶은 값을 찾는 두 가지 방법
> - **IQR**: 상위 25%~75% 범위에서 1.5배 벗어나면 이상치 (보수적)
> - **Z-score**: 평균에서 표준편차 3배 이상 떨어지면 이상치 (통계적)

둘 다 쓰면 교집합 = 확실한 이상치!

In [ ]:
def detect_iqr(series, factor=1.5):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return (series < q1 - factor * iqr) | (series > q3 + factor * iqr)

def detect_z(series, threshold=3.0):
    z = (series - series.mean()) / series.std()
    return z.abs() > threshold

z_outliers = detect_z(ffill_df)
iqr_outliers = detect_iqr(ffill_df)

print("Z-score 이상치:")
print(z_outliers.sum())
print("\nIQR 이상치:")
print(iqr_outliers.sum())
print("\n교집합 (확실한 이상치):")
print((z_outliers & iqr_outliers).sum())

### 1-7. Feature Engineering: 수익률, 변동성, 로그수익률

> 비유: 원재료(종가)에서 "양념"(파생 지표)을 만드는 단계.  
> - `pct_change()` = 일간 수익률 (어제 대비 오늘 몇 % 올랐나)
> - `rolling(N).std()` = N일 변동성 (최근 N일간 얼마나 출렁였나)
> - `log(today/yesterday)` = 로그수익률 (수학적으로 더 정확, 연속 복리 가정)

In [ ]:
features = ffill_df[['Close']].copy()
features['return'] = features['Close'].pct_change()
features['vol_5d'] = features['return'].rolling(5).std()
features['vol_20d'] = features['return'].rolling(20).std()
features['log_return'] = np.log(features['Close'] / features['Close'].shift(1))
features.dropna().head()

In [ ]:
# 전처리 파이프라인: 결측치 -> 이상치 -> feature 추가를 한번에
def preprocess_pipeline(df):
    result = df[['Close']].copy()
    # 1. 결측값 처리 (linear interpolation + ffill/bfill)
    result['Close'] = result['Close'].interpolate(method='linear')
    result['Close'] = result['Close'].ffill().bfill()
    # 2. 이상치를 ffill (IQR * 1.5)
    ret = result['Close']
    q1, q3 = ret.quantile(0.25), ret.quantile(0.75)
    iqr = q3 - q1
    outlier_mask = (ret < q1 - 1.5 * iqr) | (ret > q3 + 1.5 * iqr)
    result.loc[outlier_mask, 'Close'] = np.nan
    result['Close'] = result['Close'].ffill()
    # 3. feature 추가
    result['return'] = result['Close'].pct_change()
    result['log_return'] = np.log(result['Close'] / result['Close'].shift(1))
    return result

processed = preprocess_pipeline(price_df)
processed.head()

### 1-8. 이동평균 & 트렌드 분해

> 비유: 시계열 데이터 = **추세(Trend)** + **계절성(Seasonality)** + **잔차(Residual)**
> - 추세: "장기적으로 매출이 올라가고 있다"
> - 계절성: "여름에 잘 팔리고 겨울엔 안 팔린다"
> - 잔차: 설명 안 되는 나머지 (노이즈)

이동평균으로 노이즈를 줄인다:
- **SMA**(Simple Moving Average): 최근 N개의 단순 평균
- **EMA**(Exponential Moving Average): 최근 값에 더 큰 가중치

In [ ]:
# !pip install statsmodels  # 설치 안 되어 있으면 주석 해제
from statsmodels.tsa.seasonal import seasonal_decompose

close = ffill_df['Close']
sma_20 = close.rolling(20).mean()
ema_20 = close.ewm(span=20).mean()

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

# 상단: 이동평균
ax = axes[0]
ax.plot(close.index, close, alpha=0.4, linewidth=0.5, label='original')
ax.plot(sma_20.index, sma_20, linewidth=1.5, label='SMA(20)', color='red')
ax.plot(ema_20.index, ema_20, linewidth=1.5, label='EMA(20)', color='green')
ax.set_title('이동평균 비교')
ax.legend()

# 하단: 트렌드 분해
weekly = close.resample('W').last().dropna()
decomp = seasonal_decompose(weekly, model='additive', period=13)
axes[1].plot(decomp.trend.index, decomp.trend, label='trend', linewidth=1.5, color='blue')
axes[1].plot(decomp.seasonal.index, decomp.seasonal, label='seasonal', linewidth=1.5, color='red')
axes[1].set_title('Trend Decomposition')
axes[1].legend()

plt.tight_layout()
plt.show()

### 1-9. 여러 ETF 비교: 누적수익률 & 상관관계

> 비유: 상관관계가 낮은 ETF끼리 묶으면 = 달걀을 여러 바구니에 나누는 효과.  
> 하나가 떨어져도 다른 게 버텨준다. (분산투자)

In [ ]:
tickers = {
    '069500': 'KODEX 200 (국내주식)',
    '360750': 'TIGER S&P500 (미국주식)',
    '152380': 'KODEX 국고채10년 (채권)',
    '132030': 'KODEX 골드선물 (원자재)',
}

# 여러 ETF 종가 수집
multi_close = pd.DataFrame()
for ticker, name in tickers.items():
    df = fdr.DataReader(ticker, start=(datetime.now() - timedelta(days=365)).strftime('%Y-%m-%d'))
    multi_close[name] = df['Close']

multi_close = multi_close.dropna()
returns = multi_close.pct_change().dropna()
cum_returns = (1 + returns).cumprod()

In [ ]:
# 누적 수익률 비교 차트
fig, ax = plt.subplots(figsize=(12, 5))
for col in cum_returns.columns:
    ax.plot(cum_returns.index, cum_returns[col], label=col, linewidth=1.2)
ax.axhline(1.0, color='black')
ax.legend(fontsize=8)
ax.set_title('Cumulative Returns Comparison')
plt.show()

In [ ]:
# 상관관계 히트맵 + 60일 롤링 상관관계
corr_matrix = returns.corr()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 히트맵
im = ax1.imshow(corr_matrix, cmap='RdYlBu_r', vmin=-1, vmax=1)
ax1.set_xticks(range(len(corr_matrix)))
ax1.set_yticks(range(len(corr_matrix)))
short_names = [c.split('(')[0].strip() for c in corr_matrix.columns]
ax1.set_xticklabels(short_names, rotation=45)
ax1.set_yticklabels(short_names, fontsize=8)
for i in range(len(corr_matrix)):
    for j in range(len(corr_matrix)):
        ax1.text(j, i, f'{corr_matrix.iloc[i,j]:.2f}', fontsize=10, ha='center', va='center')
fig.colorbar(im, ax=ax1)
ax1.set_title('Correlation Matrix')

# 60일 롤링 상관관계
cols = returns.columns.tolist()
if len(cols) >= 2:
    rolling_corr = returns[cols[0]].rolling(60).corr(returns[cols[1]])
    ax2.plot(rolling_corr.index, rolling_corr, linewidth=0.8)
    ax2.set_title(f'60d Rolling Corr: {short_names[0]} vs {short_names[1]}')

plt.tight_layout()
plt.show()

### 1-10. 성과 지표 계산: 연환산 수익률, 변동성, MDD, Sharpe

> 비유:
> - **연환산 수익률**: 1년 동안 얼마 벌었나 (월급처럼 연봉 환산)
> - **변동성**: 롤러코스터 정도 (작을수록 안정적)
> - **MDD** (Maximum Drawdown): 최고점에서 최대 몇 % 빠졌나 (최악의 시나리오)
> - **Sharpe**: (수익 - 무위험금리) / 변동성 = "위험 대비 얼마나 잘 벌었나" (높을수록 좋음)

In [ ]:
def calculate_metrics(returns_series, risk_free=0.035):
    """수익률 시리즈 -> 연환산 수익률, 변동성, MDD, 샤프비율"""
    ann_ret = (1 + returns_series.mean()) ** 252 - 1
    ann_vol = returns_series.std() * np.sqrt(252)
    sharpe = (ann_ret - risk_free) / ann_vol
    cum = (1 + returns_series).cumprod()
    mdd = ((cum - cum.cummax()) / cum.cummax()).min()
    return {
        'yearly return': f'{ann_ret * 100:.1f}%',
        'yearly volatile': f'{ann_vol * 100:.1f}%',
        'MDD': f'{mdd * 100:.1f}%',
        'sharpe': f'{sharpe:.1f}'
    }

# 모든 ETF에 대해 지표 계산
metrics_rows = []
for col in returns.columns:
    m = calculate_metrics(returns[col])
    m['ETF'] = col.split('(')[0].strip()
    metrics_rows.append(m)

metrics_df = pd.DataFrame(metrics_rows).set_index('ETF')
metrics_df

### 1-11. LLM으로 분석 인사이트 생성

> 수치 기반 지표를 LLM에게 전달해서 자연어 인사이트를 받는다.  
> 두 가지 방식: (1) 먼저 통계 계산 -> LLM은 해석만 / (2) LLM에게 가격 데이터를 통째로 주고 직접 분석

In [ ]:
# 방식 1: 계산된 지표 -> LLM 해석 (더 정확)
def generate_analysis_insight(metrics_df):
    response = ChatOpenAI(model='gpt-4o-mini', temperature=0.3, max_tokens=400).invoke([
        {'role': 'system', 'content': '당신은 금융 데이터 분석가입니다. 수치 기반으로 직관적인 인사이트를 제공합니다'},
        {'role': 'user', 'content': f"""아래 ETF 비교 데이터를 분석하여 투자 인사이트를 작성하세요.

{metrics_df.to_string()}

다음 형식으로 답하세요:
1. 핵심 발견(3줄)
2. 위험 요인(2줄)
3. 분산 투자 제안(2줄)"""}
    ])
    return response.content

insight = generate_analysis_insight(metrics_df)
print(insight)

In [ ]:
# 방식 2: 통계 지표 먼저 계산 -> LLM은 요약만 (하이브리드, 더 정확)
def generate_analysis_report(ticker_name, close_prices):
    """통계 지표를 Python으로 계산한 뒤, LLM에게 해석만 맡기는 방식"""
    returns_s = close_prices.pct_change().dropna()
    sma_20 = close_prices.rolling(20).mean().iloc[-1]
    sma_60 = close_prices.rolling(60).mean().iloc[-1]
    current = close_prices.iloc[-1]
    roll_mean = returns_s.rolling(30).mean()
    roll_std = returns_s.rolling(30).std()
    z_latest = ((returns_s - roll_mean) / roll_std).iloc[-1]

    from scipy.stats import skew, kurtosis
    sk = skew(returns_s.dropna())
    kt = kurtosis(returns_s.dropna())
    vol_recent = returns_s[-20:].std() * np.sqrt(252) * 100
    vol_overall = returns_s.std() * np.sqrt(252) * 100

    indicators = {
        '현재가': f'{current:,.0f}',
        'SMA(20)': f'{sma_20:,.0f}',
        'SMA(60)': f'{sma_60:,.0f}',
        '추세': '상승' if sma_20 > sma_60 else '하락',
        '최근 롤링 Z-score': f'{z_latest:.2f}',
        '이상 여부': '정상' if abs(z_latest) < 2 else '주의' if abs(z_latest) < 3 else '이상',
        'Skewness': f'{sk:.3f}',
        'Kurtosis': f'{kt:.3f}',
        '최근 20일 변동성': f'{vol_recent:.1f}%',
        '전체 변동성': f'{vol_overall:.1f}%',
        '변동성 변화': '확대' if vol_recent > vol_overall else '축소',
    }

    response = ChatOpenAI(model='gpt-4o-mini', temperature=0.5, max_tokens=400).invoke([
        {'role': 'system', 'content': '당신은 데이터 분석가입니다. 수치에 근거해서 분석하세요'},
        {'role': 'user', 'content': f"""이 시계열 분석 지표를 바탕으로 {ticker_name}의 현재 상태를 2~3문장으로 요약해주세요

{json.dumps(indicators, ensure_ascii=False, indent=2)}

투자 추천은 하지 마세요. 객관적 현황만 서술해주세요."""}
    ])
    return response.content

print(generate_analysis_report('KODEX 200', close))

### 1-12. 데이터프레임 심화: merge, concat, groupby, pivot_table

> 비유: 엑셀의 VLOOKUP = `pd.merge`, 시트 합치기 = `pd.concat`, 피벗테이블 = `pd.pivot_table`

In [ ]:
# meta + analysis 테이블을 merge (inner join)
meta = pd.DataFrame([
    {'ticker': '069500', 'name': 'KODEX 200', 'category': '국내주식', 'expense': 0.15},
    {'ticker': '360750', 'name': 'TIGER S&P500', 'category': '해외주식', 'expense': 0.07},
    {'ticker': '152380', 'name': 'KODEX 국고채10년', 'category': '채권', 'expense': 0.05},
    {'ticker': '132030', 'name': 'KODEX 골드선물', 'category': '원자재', 'expense': 0.68},
])

analysis = pd.DataFrame([
    {'ticker': '069500', 'sharpe': 0.82, 'mdd': -0.15},
    {'ticker': '360750', 'sharpe': 1.25, 'mdd': -0.12},
    {'ticker': '152380', 'sharpe': 0.31, 'mdd': -0.05},
    {'ticker': '132030', 'sharpe': 0.95, 'mdd': -0.08},
])

# inner join (교집합) / outer (합집합) / left / right
merged = pd.merge(meta, analysis, on='ticker', how='inner')
merged

In [ ]:
# Long format 데이터 만들기 (groupby, pivot_table 실습용)
kodex = fdr.DataReader('069500', start=(datetime.now() - timedelta(days=180)).strftime('%Y-%m-%d'))
tiger = fdr.DataReader('360750', start=(datetime.now() - timedelta(days=180)).strftime('%Y-%m-%d'))
bond = fdr.DataReader('152380', start=(datetime.now() - timedelta(days=180)).strftime('%Y-%m-%d'))

price_dict = {'069500': kodex, '360750': tiger, '152380': bond}

long_data = []
for ticker, name in [('069500', 'KODEX200'), ('360750', 'TIGER_SP500'), ('152380', 'KODEX_국고채')]:
    df = price_dict.get(ticker)
    if df is None: continue
    ret = df['Close'].pct_change().dropna()
    for date, val in ret.items():
        long_data.append({
            'date': date, 'ticker': ticker, 'name': name,
            'return': val, 'weekday': date.strftime('%a'), 'month': date.month,
        })

long_df = pd.DataFrame(long_data)
print(f"Long format: {long_df.shape}")
long_df.head()

In [ ]:
# groupby: 엑셀의 그룹별 통계
summary = long_df.groupby('name')['return'].agg(
    평균수익률='mean', 변동성='std', 최대='max', 최소='min', 거래일수='count'
)
summary['평균수익률'] = (summary['평균수익률'] * 100).round(4)
summary

In [ ]:
# pivot_table: 엑셀 피벗테이블과 동일 (월별 평균 수익률)
pd.pivot_table(long_df, values='return', index='name', columns='month', aggfunc='mean')

In [ ]:
# 월별 변동성 피벗 -> stack으로 최대 변동성 찾기
vol_pivot = pd.pivot_table(long_df, values='return', index='name', columns='month', aggfunc='std')
stacked = vol_pivot.stack()
print(f"최대 변동성: {stacked.idxmax()} = {stacked.max():.4f}")

---
# Part 2: RAG용 Document 생성 & 검색 시스템 구축

> Part 1에서 손질한 데이터를 가지고 이제 진짜 RAG를 만든다.  
> 
> **왜 단순한 ETF 정보만으로는 부족한가?**  
> - "돈 벌고 싶은데 뭐 사야 돼?" --> 키워드(코스피, 대형주...)에 안 걸림  
> - 벡터 서치만으로는 비슷비슷한 description 1000개 중에서 정확히 골라내기 어려움  
> - **해결**: (1) LLM으로 풍부한 description 생성 (2) BM25 키워드 검색 추가 (3) 메타데이터 필터링

비유: 벡터 서치 = 그물로 물고기 잡기. 그물 하나로는 원하는 물고기만 잡기 어렵다.  
그물 여러 개(벡터 + 키워드 + 메타데이터)를 던지면 그 중 하나는 걸린다.

In [ ]:
import os, json, time, re, math
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from collections import Counter
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS as LangchainFAISS
from langchain_core.documents import Document

# load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
MODEL = "gpt-4o-mini"
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

### 2-1. ETF 목록 수집 & ticker-name 매핑

In [ ]:
import FinanceDataReader as fdr

etf_listing = fdr.StockListing('ETF/KR')
print(f"상장 ETF 수: {len(etf_listing)}")

# ticker -> name 딕셔너리 생성 (6자리 zero-fill)
name_by_ticker = dict(zip(
    etf_listing['Symbol'].astype(str).str.zfill(6),
    etf_listing['Name']
))

# 확인
print(f"069500 -> {name_by_ticker.get('069500')}")
print(f"360750 -> {name_by_ticker.get('360750')}")

### 2-2. 분석 대상 ETF 선정 (12종)

> 카테고리(국내주식, 해외주식, 채권, 원자재, 부동산, 혼합, 테마, 배당)를 골고루 포함하도록 12개 선정.  
> 실제 서비스에서는 1000개+ 전체를 넣어도 됨. 스코프에 따라 결정.

In [ ]:
tickers_info = {
    "069500": {"category": "국내주식", "expense_ratio": 0.15, "dividend_yield": 1.8,
               "keywords": ["코스피", "대형주", "인덱스", "분산투자", "국내주식"]},
    "379800": {"category": "해외주식", "expense_ratio": 0.05, "dividend_yield": 0.0,
               "keywords": ["미국", "S&P500", "대형주", "성장", "해외주식"]},
    "411060": {"category": "배당",     "expense_ratio": 0.01, "dividend_yield": 3.5,
               "keywords": ["미국", "배당", "배당성장", "저비용", "안정"]},
    "305540": {"category": "테마",     "expense_ratio": 0.45, "dividend_yield": 0.0,
               "keywords": ["2차전지", "배터리", "테마", "성장", "고위험"]},
    "379810": {"category": "해외주식", "expense_ratio": 0.07, "dividend_yield": 0.0,
               "keywords": ["미국", "나스닥", "기술주", "성장", "IT"]},
    "381180": {"category": "해외주식", "expense_ratio": 0.49, "dividend_yield": 0.0,
               "keywords": ["반도체", "AI", "미국", "기술주", "고위험"]},
    "132030": {"category": "원자재",   "expense_ratio": 0.68, "dividend_yield": 0.0,
               "keywords": ["금", "원자재", "인플레이션", "헤지", "안전자산"]},
    "167860": {"category": "채권",     "expense_ratio": 0.15, "dividend_yield": 2.8,
               "keywords": ["채권", "국고채", "안전", "이자", "저위험"]},
    "214980": {"category": "채권",     "expense_ratio": 0.05, "dividend_yield": 3.2,
               "keywords": ["단기채", "안전", "예금대안", "저위험", "이자"]},
    "329200": {"category": "부동산",   "expense_ratio": 0.29, "dividend_yield": 4.2,
               "keywords": ["리츠", "부동산", "배당", "인프라", "실물자산"]},
    "284430": {"category": "혼합",     "expense_ratio": 0.09, "dividend_yield": 1.5,
               "keywords": ["혼합", "자산배분", "균형", "중위험", "분산투자"]},
    "371460": {"category": "해외주식", "expense_ratio": 0.49, "dividend_yield": 0.0,
               "keywords": ["중국", "전기차", "테마", "해외주식", "고위험"]},
}

for t, info in tickers_info.items():
    name = name_by_ticker.get(t, "(이름 미확인)")
    print(f" {t} | {name:30s} | {info['category']}")

### 2-3. 가격 수집 + 지표 계산 + 위험도 분류

In [ ]:
def collect_etf_price(ticker, days=365):
    """FinanceDataReader로 가격 데이터 수집"""
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - timedelta(days=days)).strftime('%Y-%m-%d')
    df = fdr.DataReader(ticker, start_date, end_date)
    return {'status': 'real', 'data': df, 'ticker': ticker}

def compute_metrics(df):
    """Close -> 연환산 수익률, 변동성, MDD 계산"""
    close = df['Close'].ffill()
    ret = close.pct_change().dropna()
    ann_ret = ((1 + ret.mean()) ** 252 - 1) * 100
    ann_vol = ret.std() * np.sqrt(252) * 100
    cum = (1 + ret).cumprod()
    mdd = ((cum - cum.cummax()) / cum.cummax()).min() * 100
    return {'return_1y': round(ann_ret, 2), 'volatility': round(ann_vol, 2), 'mdd': round(mdd, 2)}

def risk_from_vol(vol):
    """변동성 기반 위험도 분류"""
    if vol < 5:  return "낮음"
    if vol < 15: return "약간 낮음"
    if vol < 25: return "중간"
    return "높음"

In [ ]:
# 12개 ETF 데이터 수집 + 지표 계산 (1~2분 소요)
etf_data = []
for t, info in tickers_info.items():
    result = collect_etf_price(t, days=365)
    metrics = compute_metrics(result['data'])
    name = name_by_ticker.get(t, f"ETF_{t}")
    etf_data.append({
        "ticker": t,
        "name": name,
        **info,          # category, expense_ratio, dividend_yield, keywords
        **metrics,       # return_1y, volatility, mdd
        'risk_level': risk_from_vol(metrics['volatility']),
        'data_status': result['status']
    })

print(f"수집 완료: {len(etf_data)}개")
etf_data[:2]

### 2-4. LLM으로 Description 생성 -> Document 구성

> **핵심 아이디어**: 숫자 데이터(수익률, 변동성 등)를 LLM이 자연어 설명으로 변환.  
> 이렇게 하면 벡터 임베딩이 의미를 더 잘 포착한다.
>
> 비유: 도서관 카드(메타데이터) + 책 요약문(description) = 검색하기 쉬운 Document

In [ ]:
def generate_description(etf):
    """ETF 정보를 LLM에게 전달해서 자연어 설명문 생성"""
    prompt = f"""다음 ETF의 특징을 한국어 1~2문장으로 간결히 설명하세요.
    이름 : {etf['name']}
    카테고리 : {etf['category']}
    키워드 : {','.join(etf['keywords'])}
    수수료 : {etf['expense_ratio']}% / 배당수익률: {etf['dividend_yield']}%
    1년수익률 : {etf['return_1y']}% / 변동성: {etf['volatility']}% / MDD : {etf['mdd']}%
    """
    return llm.invoke([{'role': 'user', 'content': prompt}]).content.strip()

In [ ]:
# Document 리스트 생성 (LLM 호출 12회, 1~2분 소요)
documents = []
for etf in etf_data:
    desc = generate_description(etf)
    # page_content: 벡터 임베딩에 사용될 텍스트
    text = (
        f"{etf['name']} ({etf['category']}): {desc} "
        f"키워드: {', '.join(etf['keywords'])} "
        f"수수료 {etf['expense_ratio']}%, 배당수익률: {etf['dividend_yield']}% "
        f"수익률: {etf['return_1y']}% / 변동성: {etf['volatility']}% / MDD: {etf['mdd']}%"
    )
    # metadata: 필터링에 사용 (숫자 필드 포함)
    metadata = {k: v for k, v in etf.items() if k != "keywords"}
    metadata['keywords'] = ', '.join(etf['keywords'])
    documents.append(Document(page_content=text, metadata=metadata))

print(f"Document 수: {len(documents)}")
print(f"\n[예시] {documents[0].metadata['name']}")
print(documents[0].page_content[:200])

### 2-5. FAISS 벡터 스토어 구축 & 벡터 검색 테스트

In [ ]:
# Document -> FAISS 벡터 스토어
vectorstore = LangchainFAISS.from_documents(documents, embeddings)
print(f"벡터 스토어 인덱스 수: {vectorstore.index.ntotal}")

# 벡터 검색 테스트
query = '미국 기술주에 투자하고 싶어요'
results = vectorstore.similarity_search_with_score(query, k=3)
for doc, score in results:
    print(f"  [{score:.3f}] {doc.metadata['name']}")

### 2-6. LLM으로 합성 질문 생성 (RAG 평가용)

> 서비스 오픈 전에 "사용자가 이런 질문을 하겠지"를 미리 만들어 검색 품질을 테스트.  
> LLM한테 ETF 목록을 주고 "투자자가 물어볼 법한 질문 5개 생성"을 요청.

In [ ]:
etf_knowledge_base = [
    {"ticker": "069500", "name": "KODEX 200", "category": "국내주식",
     "description": "KOSPI 200 지수 추적. 수수료 0.15%. 대형주 중심 분산투자.",
     "risk": "중간", "expense_ratio": 0.15},
    {"ticker": "379800", "name": "KODEX 미국S&P500TR", "category": "해외주식",
     "description": "S&P500 추적, 배당 자동 재투자. 수수료 0.05%.",
     "risk": "중간", "expense_ratio": 0.05},
    {"ticker": "461460", "name": "KODEX 미국나스닥100TR", "category": "해외주식",
     "description": "나스닥100 기술주 중심. 높은 성장성/변동성. 수수료 0.05%.",
     "risk": "높음", "expense_ratio": 0.05},
]

etf_names = [e['name'] + ': ' + e['description'] for e in etf_knowledge_base]

prompt = f"""다음 ETF 데이터를 보고 실제 투자자가 물어볼 법한 질문 5개를 생성하세요

ETF 목록:
{json.dumps(etf_names, ensure_ascii=False, indent=2)}

형식 : 번호. 질문"""

synthetic = llm.invoke(prompt)
print(synthetic.content)

### 2-7. BM25 키워드 검색 (한국어 형태소 분석기 Kiwi 활용)

> **벡터 서치의 한계**: "KODEX 200" 같은 정확한 이름은 키워드 매칭이 더 빠르고 정확.  
> 벡터는 의미적 뉘앙스는 잘 잡지만, 정확한 단어 매칭에는 약함.
>
> **BM25** = TF-IDF의 개선판
> - k1 파라미터: TF(단어 빈도)의 영향력 조절 (높으면 빈도 차이 더 중시)
> - b 파라미터: 문서 길이 정규화 (높으면 긴 문서에 불리)
>
> **Kiwi 형태소 분석기**: 한국어를 명사/동사 등으로 쪼개서 "은/는/이/가" 같은 조사를 제거.  
> 예) "배당수익률 3% 이상인 미국 ETF" -> ['배당', '수익', '3', '이상', '미국', 'etf']

In [ ]:
# !pip install kiwipiepy rank_bm25  # 설치 안 되어 있으면 주석 해제
from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi

kiwi = Kiwi()

In [ ]:
def kiwi_tokenize(text):
    """한국어 형태소 분석: 명사/고유명사/영어/숫자만 추출"""
    tokens = kiwi.tokenize(text)
    result = []
    for token in tokens:
        # SL=영어, SN=숫자, NNG=일반명사, NNP=고유명사
        if token.tag in ("SL", "SN", "NNG", "NNP"):
            result.append(token.form.lower())
    return result

# 테스트
print(kiwi_tokenize("KODEX 200 ETF는 어떤 종류의 주식에 투자하나요?"))
print(kiwi_tokenize("배당수익률 3% 이상인 미국 ETF를 추천해주세요"))

In [ ]:
# Document corpus를 토큰화 -> BM25 인덱스 구축
corpus = [kiwi_tokenize(doc.page_content) for doc in documents]
bm25 = BM25Okapi(corpus)

def bm25_search(query, k=5):
    """BM25 키워드 검색 (score > 0인 것만 반환)"""
    tokens = kiwi_tokenize(query)
    scores = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:k]
    return [(documents[i], scores[i]) for i in top_idx if scores[i] > 0]

In [ ]:
# 벡터 검색 vs BM25 키워드 검색 비교
test_queries = [
    "KODEX 200",
    "안전한 투자 상품",
    "배당 ETF",
    "인플레이션 방어"
]

print(f"{'쿼리':20s} | {'벡터 검색':25s} | {'BM25 키워드':25s}")
print("-" * 75)
for q in test_queries:
    v_result = vectorstore.similarity_search(q, k=1)
    b_result = bm25_search(q, k=1)
    v_name = v_result[0].metadata['name'] if v_result else "-"
    b_name = b_result[0][0].metadata['name'] if b_result else "-"
    print(f"{q:20s} | {v_name:25s} | {b_name:25s}")

### 2-8. BM25 하이퍼파라미터 (k1, b) 실험

> - **k1** (기본 1.5): TF의 영향력. 높으면 빈도 차이를 더 중시
> - **b** (기본 0.75): 문서 길이 정규화. 높으면 긴 문서에 더 불리
>
> TF-IDF의 한계를 BM25가 해결:
> 1. TF 값이 무한대로 올라가는 것에 cap을 씌움
> 2. 50번 나온 단어와 100번 나온 단어의 차이를 완화
> 3. 문서 길이를 고려한 normalize

In [ ]:
query = "배당 ETF"
tokens = kiwi_tokenize(query)

params = [(0.5, 0.25), (1.5, 0.75), (2.5, 0.9)]
for k1, b in params:
    bm25_temp = BM25Okapi(corpus, k1=k1, b=b)
    scores = bm25_temp.get_scores(tokens)
    top3 = np.argsort(scores)[::-1][:3]
    print(f"k1={k1}, b={b}")
    for idx in top3:
        name = documents[idx].metadata['name']
        print(f"  [{scores[idx]:.3f}] {name}")
    print()

### 2-9. 메타데이터 필터링 검색

> 벡터 검색으로 후보 20개를 먼저 뽑고, 메타데이터(카테고리, 수수료 등)로 필터링해서 최종 K개를 반환.  
>
> 비유: 백화점에서 먼저 "여성복 코너"로 가고(벡터 = 대략적 방향),  
> 그 다음 "가격 10만원 이하"로 좁힌다(메타데이터 = 정확한 조건).
>
> 필터 조건은 딕셔너리로 전달:
> - 단순 일치: `{"category": "해외주식"}`
> - 범위 조건: `{"expense_ratio": {"less_than": 0.5}}`

In [ ]:
def filtered_search(vectorstore, query, filters=None, k=5, fetch_k=20):
    """
    벡터 검색 + 메타데이터 필터링
    filters 예시: {"category": "해외주식", "expense_ratio": {"less_than": 0.5}}
    """
    results = vectorstore.similarity_search_with_score(query, fetch_k)
    if not filters:
        return results[:k]

    filtered = []
    for doc, score in results:
        match = True
        for key, condition in filters.items():
            val = doc.metadata.get(key)
            if isinstance(condition, dict):
                # 범위 조건: {"less_than": 0.5} 또는 {"greater_than": 10}
                if "less_than" in condition and val > condition["less_than"]:
                    match = False
                if "greater_than" in condition and val < condition["greater_than"]:
                    match = False
            elif val != condition:
                # 단순 일치 조건
                match = False
        if match:
            filtered.append((doc, score))

    return filtered[:k]

In [ ]:
# 테스트: 해외주식 카테고리 + 수수료 0.5% 미만
results = filtered_search(
    vectorstore, "저비용 해외 ETF",
    filters={"category": "해외주식", "expense_ratio": {"less_than": 0.5}},
    k=3
)

print("필터링 결과 (카테고리=해외주식, 수수료<0.5%):")
for doc, score in results:
    print(f"  [{score:.3f}] {doc.metadata['name']} "
          f"(카테고리={doc.metadata['category']}, "
          f"수수료={doc.metadata['expense_ratio']}%)")

---
## 오늘 수업 정리

### Part 1 (데이터 전처리)
| 단계 | 핵심 | 비유 |
|------|------|------|
| 결측치 처리 | ffill, interpolation, rolling mean | 출석부 빈칸 채우기 |
| 이상치 탐지 | IQR, Z-score | 시험 점수 이상한 애 찾기 |
| Feature Engineering | return, volatility, log_return | 원재료에서 양념 만들기 |
| 이동평균/분해 | SMA, EMA, seasonal_decompose | 노이즈 제거 + 추세 분리 |
| 성과 지표 | Sharpe, MDD | 위험 대비 보상 계산 |

### Part 2 (RAG Document 구축)
| 단계 | 핵심 | 비유 |
|------|------|------|
| LLM Description 생성 | 숫자 데이터 -> 자연어 | 도서 요약문 작성 |
| FAISS 벡터 스토어 | Document 임베딩 저장 | 도서관 서가 배치 |
| BM25 + Kiwi | 한국어 형태소 키워드 검색 | 카드 카탈로그 |
| 메타데이터 필터링 | 카테고리/수수료 등 조건 검색 | 쇼핑몰 필터 |
| 합성 질문 생성 | LLM으로 테스트 쿼리 자동 생성 | 모의고사 출제 |

### Trade-off (항상 기억할 것)
- Description 짧게 -> 키워드/벡터 검색 잘 걸림, but 답변 풍부하지 않음
- Description 길게 -> 답변 풍부, but 비슷한 문서끼리 구분 어려움
- 벡터 검색만 -> 뉘앙스 잡지만 정확한 이름 매칭 약함
- 키워드 검색만 -> 이름 매칭 강하지만 의미 파악 약함
- **해결**: 여러 그물을 던져라! (벡터 + 키워드 + 메타데이터)

> 내일(260417): 이 노트북에 이어서 금융 ETF 챗봇 완성 (RAG 파이프라인 + Gradio UI)